<a href="https://colab.research.google.com/github/dranphphmithe-ux/Book-Rental-System-Project/blob/Nam-Wan/%E0%B8%AA%E0%B8%B3%E0%B9%80%E0%B8%99%E0%B8%B2%E0%B8%82%E0%B8%AD%E0%B8%87_book_rental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datetime import datetime

class Receipt:
    def __init__(self, order_id: str, customer, items: list, days_rented: int, days_late: int = 0):
        """
        :param order_id: รหัสใบเสร็จ/การยืม
        :param customer: Object ของผู้ใช้ (เช่น LibraryOrder)
        :param items: รายชื่อ Object หนังสือที่ยืม [Book1, Book2, ...]
        :param days_rented: จำนวนวันที่ต้องการยืมจริงตามสัญญา
        :param days_late: จำนวนวันที่ส่งคืนช้า ( default = 0 )
        """
        self.order_id = order_id
        self.customer = customer
        self.items = items
        self.days_rented = days_rented
        self.days_late = max(0, days_late)
        self.date_issued = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def _calculate_book_rental_fee(self, days: int) -> float:
        """คำนวณค่ายืมต่อ 1 เล่ม: 3 วัน 20 บาท, เศษวันละ 7 บาท"""
        sets_of_3 = days // 3
        remaining_days = days % 3
        return (sets_of_3 * 20) + (remaining_days * 7)

    def calculate_totals(self):
        total_books = len(self.items)

        # 1. โปรโมชั่น: ยืมครบ 10 เล่ม ฟรี 1 เล่ม (จ่ายเงินแค่ total_books - free_books)
        free_books = total_books // 10
        chargeable_books = total_books - free_books

        # 2. คำนวณค่ายืมปกติ
        rental_fee_per_book = self._calculate_book_rental_fee(self.days_rented)
        total_rental_fee = chargeable_books * rental_fee_per_book

        # 3. คำนวณค่าปรับ (กรณีคืนช้า: 10 บาท / เล่ม / วัน)
        total_fine = total_books * self.days_late * 10

        # 4. สรุปรวมเงินทั้งหมด
        grand_total = total_rental_fee + total_fine

        # 5. คำนวณแต้มสะสม (คืนช้า = 0 แต้ม, คืนตรงเวลา = ทุกๆ 10 บาทได้ 1 แต้ม)
        earned_points = 0
        if self.days_late == 0:
            earned_points = int(total_rental_fee // 10)

        return {
            "total_books": total_books,
            "free_books": free_books,
            "chargeable_books": chargeable_books,
            "rental_fee_per_book": rental_fee_per_book,
            "total_rental_fee": total_rental_fee,
            "total_fine": total_fine,
            "grand_total": grand_total,
            "earned_points": earned_points
        }

    def print_receipt(self):
        calc = self.calculate_totals()

        # หากคืนตรงเวลา ให้เพิ่มแต้มเข้าไปที่ตัว customer
        if calc["earned_points"] > 0 and hasattr(self.customer, 'add_points'):
            self.customer.add_points(calc["earned_points"])

        print("=" * 50)
        print(f"{'ใบเสร็จรับเงิน / Receipt':^50}")
        print("=" * 50)
        print(f"เลขที่ใบเสร็จ: {self.order_id}")
        print(f"วันที่ออกใบเสร็จ: {self.date_issued}")
        print(f"ชื่อลูกค้า: {self.customer.name} (ID: {self.customer.customer_id})")
        print(f"จำนวนวันที่ยืม: {self.days_rented} วัน")
        if self.days_late > 0:
            print(f"Status: ⚠️ คืนเกินกำหนด {self.days_late} วัน")
        else:
            print(f"Status: ✅ คืนตรงตามกำหนด")
        print("-" * 50)

        print(f"รายการหนังสือที่ยืม (ทั้งหมด {calc['total_books']} เล่ม):")
        for i, book in enumerate(self.items, 1):
            title = getattr(book, 'book_title', f'หนังสือเล่มที่ {i}')
            print(f"  {i:02d}. {title}")

        print("-" * 50)
        print(f"อัตราค่ายืมต่อเล่ม ({self.days_rented} วัน): {calc['rental_fee_per_book']:.2f} บาท")

        if calc['free_books'] > 0:
            print(f"🎁 โปรโมชั่นยืมครบ 10 เล่ม: ฟรี {calc['free_books']} เล่ม (คิดเงิน {calc['chargeable_books']} เล่ม)")

        print(f"รวมค่ายืมหนังสือ: {calc['total_rental_fee']:.2f} บาท")

        if calc['total_fine'] > 0:
            print(f"❌ ค่าปรับคืนช้า ({self.days_late} วัน x {calc['total_books']} เล่ม x 10B): {calc['total_fine']:.2f} บาท")

        print("-" * 50)
        print(f"ยอดชำระสุทธิ (Grand Total): {calc['grand_total']:.2f} บาท")
        print("-" * 50)

        if self.days_late > 0:
            print("🚫 คืนช้ากว่ากำหนด: **ไม่ได้รับแต้มสะสม**")
        else:
            print(f"✨ แต้มที่ได้รับครั้งนี้: +{calc['earned_points']} แต้ม")
            print(f"🏆 แต้มสะสมรวมทั้งหมด: {self.customer.points} แต้ม")
        print("=" * 50 + "\n")

In [3]:
import csv
import io
from datetime import datetime
from urllib.request import urlopen

# ==========================================
# 1. Class LibraryOrder (จัดการสมาชิก)
# ==========================================
class LibraryOrder:
    def __init__(self, name, customer_id, phone, email, points, tier, register_date):
        self.name = name
        self.customer_id = customer_id
        self.phone = phone
        self.email = email
        self.points = int(points)
        self.tier = tier
        self.register_date = register_date
        self.status = 'รอดำเนินการ'

    def register(self):
        print(f"สมัครสมาชิกสำเร็จ: {self.register_date}")

    def login(self):
        self.status = 'เข้าสู่ระบบเรียบร้อย'
        print(f"{self.name} {self.status}")
        return True

    def update_profile(self, name: str = None, phone: str = None, email: str = None):
        if name: self.name = name
        if phone: self.phone = phone
        if email: self.email = email
        print("อัปเดตข้อมูลส่วนตัวเรียบร้อย")

    def add_points(self, amount: int):
        self.points += amount
        print(f"เพิ่ม {amount} แต้ม | แต้มรวมปัจจุบัน: {self.points}")

    def get_purchase_history(self):
        return []


# ==========================================
# 2. Class Book (จัดการข้อมูลหนังสือ)
# ==========================================
class Book:
    def __init__(self, book_isbn, book_title, book_author, price, category_id, sub_category_id, shelf_location, stock_qty):
        self.book_isbn = book_isbn
        self.book_title = book_title
        self.book_author = book_author
        self.price = float(price)  # ต้องรับเฉพาะตัวเลข เช่น 90 หรือ "90"
        self.category_id = category_id
        self.sub_category_id = sub_category_id
        self.shelf_location = shelf_location
        self.stock_qty = int(stock_qty)


# ==========================================
# 3. Class Receipt (ระบบออกใบเสร็จและคิดเงิน)
# ==========================================
class Receipt:
    def __init__(self, order_id: str, customer: LibraryOrder, items: list, days_rented: int, days_late: int = 0):
        self.order_id = order_id
        self.customer = customer
        self.items = items
        self.days_rented = days_rented
        self.days_late = max(0, days_late)
        self.date_issued = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def _calculate_book_rental_fee(self, days: int) -> float:
        """คำนวณค่ายืมต่อเล่ม: 3 วัน 20 บาท, เศษวันละ 7 บาท"""
        sets_of_3 = days // 3
        remaining_days = days % 3
        return (sets_of_3 * 20) + (remaining_days * 7)

    def calculate_totals(self):
        total_books = len(self.items)

        # โปรโมชั่น: ยืมครบ 10 เล่ม ฟรี 1 เล่ม
        free_books = total_books // 10
        chargeable_books = total_books - free_books

        # คำนวณค่ายืมปกติ
        rental_fee_per_book = self._calculate_book_rental_fee(self.days_rented)
        total_rental_fee = chargeable_books * rental_fee_per_book

        # คำนวณค่าปรับ (คืนช้า: 10 บาท / เล่ม / วัน)
        total_fine = total_books * self.days_late * 10

        # สรุปรวมเงิน
        grand_total = total_rental_fee + total_fine

        # คำนวณแต้มสะสม (คืนตรงเวลาเท่านั้น: ทุก 10 บาท = 1 แต้ม)
        earned_points = 0
        if self.days_late == 0:
            earned_points = int(total_rental_fee // 10)

        return {
            "total_books": total_books,
            "free_books": free_books,
            "chargeable_books": chargeable_books,
            "rental_fee_per_book": rental_fee_per_book,
            "total_rental_fee": total_rental_fee,
            "total_fine": total_fine,
            "grand_total": grand_total,
            "earned_points": earned_points
        }

    def print_receipt(self):
        calc = self.calculate_totals()

        # บวกแต้มเพิ่มให้ผู้ใช้กรณีคืนตรงเวลา
        if calc["earned_points"] > 0:
            self.customer.add_points(calc["earned_points"])

        print("=" * 55)
        print(f"{'ใบเสร็จรับเงิน / Receipt':^55}")
        print("=" * 55)
        print(f"เลขที่ใบเสร็จ: {self.order_id}")
        print(f"วันที่ออกใบเสร็จ: {self.date_issued}")
        print(f"ชื่อลูกค้า: {self.customer.name} (ID: {self.customer.customer_id})")
        print(f"จำนวนวันที่ยืม: {self.days_rented} วัน")
        print(f"สถานะการคืน: {'⚠️ คืนช้า ' + str(self.days_late) + ' วัน' if self.days_late > 0 else '✅ คืนตรงเวลา'}")
        print("-" * 55)

        print(f"รายการหนังสือที่ยืม ({calc['total_books']} เล่ม):")
        for i, book in enumerate(self.items, 1):
            print(f"  [{i:02d}] {book.book_title} (ราคาปก {book.price} บาท)")

        print("-" * 55)
        print(f"อัตราค่ายืมต่อเล่ม ({self.days_rented} วัน): {calc['rental_fee_per_book']:.2f} บาท")

        if calc['free_books'] > 0:
            print(f"🎁 โปรโมชั่นยืมครบ 10 เล่ม: ฟรี {calc['free_books']} เล่ม (คิดเงิน {calc['chargeable_books']} เล่ม)")

        print(f"รวมค่ายืมหนังสือ: {calc['total_rental_fee']:.2f} บาท")

        if calc['total_fine'] > 0:
            print(f"❌ ค่าปรับคืนช้า ({self.days_late} วัน x {calc['total_books']} เล่ม x 10B): {calc['total_fine']:.2f} บาท")

        print("-" * 55)
        print(f"ยอดชำระสุทธิ (Grand Total): {calc['grand_total']:.2f} บาท")
        print("-" * 55)

        if self.days_late > 0:
            print("🚫 คืนช้ากว่ากำหนด: ไม่ได้รับแต้มสะสม")
        else:
            print(f"✨ แต้มที่ได้รับครั้งนี้: +{calc['earned_points']} แต้ม")
            print(f"🏆 แต้มสะสมรวมทั้งหมด: {self.customer.points} แต้ม")
        print("=" * 55 + "\n")


# ==========================================
# 4. ทดลองรันใช้งานระบบ
# ==========================================

# 1. สร้างวัตถุหนังสือ (แก้ไขราคาเป็นตัวเลข pure number)
book1 = Book(
    book_isbn="B46010101",
    book_title="ผ่าพิภพไททัน / Attack on Titan",
    book_author="Hajime Isayama",
    price=90,  # แก้จาก "90 บาท/เล่ม" เป็น 90
    category_id="01",
    sub_category_id="01",
    shelf_location="A-01",
    stock_qty=34
)

# 2. สร้างวัตถุสมาชิก
user1 = LibraryOrder(
    name="สมชาย ใจดี",
    customer_id="C001",
    phone="081-234-5678",
    email="somchai@email.com",
    points=50,
    tier="Gold",
    register_date="2026-01-01"
)

# 3. จำลองการยืมหนังสือ 11 เล่ม (เพื่อทดสอบโปรโมชั่น ยืม 10 ฟรี 1)
cart_items = [book1] * 11

# --- ทดสอบกรณีที่ 1: ยืม 7 วัน และคืนตรงเวลา ---
receipt1 = Receipt(
    order_id="REC-2026001",
    customer=user1,
    items=cart_items,
    days_rented=7,
    days_late=0
)
receipt1.print_receipt()

# --- ทดสอบกรณีที่ 2: ยืม 3 วัน แต่คืนช้าไป 2 วัน ---
receipt2 = Receipt(
    order_id="REC-2026002",
    customer=user1,
    items=cart_items,
    days_rented=3,
    days_late=2
)
receipt2.print_receipt()

เพิ่ม 47 แต้ม | แต้มรวมปัจจุบัน: 97
               ใบเสร็จรับเงิน / Receipt                
เลขที่ใบเสร็จ: REC-2026001
วันที่ออกใบเสร็จ: 2026-08-26 15:28:39
ชื่อลูกค้า: สมชาย ใจดี (ID: C001)
จำนวนวันที่ยืม: 7 วัน
สถานะการคืน: ✅ คืนตรงเวลา
-------------------------------------------------------
รายการหนังสือที่ยืม (11 เล่ม):
  [01] ผ่าพิภพไททัน / Attack on Titan (ราคาปก 90.0 บาท)
  [02] ผ่าพิภพไททัน / Attack on Titan (ราคาปก 90.0 บาท)
  [03] ผ่าพิภพไททัน / Attack on Titan (ราคาปก 90.0 บาท)
  [04] ผ่าพิภพไททัน / Attack on Titan (ราคาปก 90.0 บาท)
  [05] ผ่าพิภพไททัน / Attack on Titan (ราคาปก 90.0 บาท)
  [06] ผ่าพิภพไททัน / Attack on Titan (ราคาปก 90.0 บาท)
  [07] ผ่าพิภพไททัน / Attack on Titan (ราคาปก 90.0 บาท)
  [08] ผ่าพิภพไททัน / Attack on Titan (ราคาปก 90.0 บาท)
  [09] ผ่าพิภพไททัน / Attack on Titan (ราคาปก 90.0 บาท)
  [10] ผ่าพิภพไททัน / Attack on Titan (ราคาปก 90.0 บาท)
  [11] ผ่าพิภพไททัน / Attack on Titan (ราคาปก 90.0 บาท)
-------------------------------------------------------
อั

สุ่มรายการที่ลูกค้าเคยยืม

In [13]:
import random
from datetime import datetime

# ==========================================
# 1. Class Data Models
# ==========================================
class LibraryOrder:
    def __init__(self, name, customer_id, phone, email, points, tier, register_date):
        self.name = name
        self.customer_id = customer_id
        self.phone = phone
        self.email = email
        self.points = int(points)
        self.tier = tier
        self.register_date = register_date

    def add_points(self, amount: int):
        self.points += amount


class Book:
    def __init__(self, book_isbn, book_title, price):
        self.book_isbn = book_isbn
        self.book_title = book_title
        self.price = float(price)


class Receipt:
    def __init__(self, order_id: str, customer: LibraryOrder, items: list, days_rented: int, days_late: int = 0):
        self.order_id = order_id
        self.customer = customer
        self.items = items
        self.days_rented = days_rented
        self.days_late = max(0, days_late)
        self.date_issued = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def _calculate_book_rental_fee(self, days: int) -> float:
        sets_of_3 = days // 3
        remaining_days = days % 3
        return (sets_of_3 * 20) + (remaining_days * 7)

    def calculate_totals(self):
        total_books = len(self.items)
        free_books = total_books // 10
        chargeable_books = total_books - free_books

        rental_fee_per_book = self._calculate_book_rental_fee(self.days_rented)
        total_rental_fee = chargeable_books * rental_fee_per_book

        # ค่าปรับคืนช้า 10 บาท/เล่ม/วัน
        total_fine = total_books * self.days_late * 10
        grand_total = total_rental_fee + total_fine

        # คืนตรงเวลาได้ 1 แต้มต่อทุก 10 บาท, คืนช้าได้ 0 แต้ม
        earned_points = 0
        if self.days_late == 0:
            earned_points = int(total_rental_fee // 10)

        return {
            "total_books": total_books,
            "free_books": free_books,
            "chargeable_books": chargeable_books,
            "rental_fee_per_book": rental_fee_per_book,
            "total_rental_fee": total_rental_fee,
            "total_fine": total_fine,
            "grand_total": grand_total,
            "earned_points": earned_points
        }

    def print_receipt(self):
        calc = self.calculate_totals()

        if calc["earned_points"] > 0:
            self.customer.add_points(calc["earned_points"])

        print("=" * 60)
        print(f"{'ใบเสร็จรับเงิน / Receipt':^60}")
        print("=" * 60)
        print(f"เลขที่ใบเสร็จ: {self.order_id:<20} วันที่: {self.date_issued}")
        print(f"ชื่อลูกค้า: {self.customer.name} (ID: {self.customer.customer_id}) | ระดับ: {self.customer.tier}")
        print(f"ระยะเวลายืม: {self.days_rented} วัน | สถานะ: {'⚠️ คืนช้า ' + str(self.days_late) + ' วัน' if self.days_late > 0 else '✅ คืนตรงเวลา'}")
        print("-" * 60)

        print(f"รายการหนังสือที่ยืม ({calc['total_books']} เล่ม):")
        for i, book in enumerate(self.items, 1):
            print(f"  [{i:02d}] {book.book_isbn} - {book.book_title}")

        print("-" * 60)
        print(f"ค่ายืมต่อเล่ม ({self.days_rented} วัน): {calc['rental_fee_per_book']:.2f} บาท")
        if calc['free_books'] > 0:
            print(f"🎁 โปรโมชั่นยืมครบ 10 เล่ม: ฟรี {calc['free_books']} เล่ม (คิดเงิน {calc['chargeable_books']} เล่ม)")
        print(f"รวมค่ายืมหนังสือ: {calc['total_rental_fee']:.2f} บาท")

        if calc['total_fine'] > 0:
            print(f"❌ ค่าปรับคืนช้า ({self.days_late} วัน x {calc['total_books']} เล่ม x 10B): {calc['total_fine']:.2f} บาท")

        print("-" * 60)
        print(f"ยอดชำระสุทธิ (Grand Total): {calc['grand_total']:.2f} บาท")

        if self.days_late > 0:
            print("🚫 คืนช้ากว่ากำหนด: ไม่ได้รับแต้มสะสม")
        else:
            print(f"✨ แต้มที่ได้รับ: +{calc['earned_points']} แต้ม | แต้มสะสมรวม: {self.customer.points} แต้ม")
        print("=" * 60 + "\n")


# ==========================================
# 2. คลังข้อมูลสำหรับสุ่ม
# ==========================================
first_names = ["สมชาย", "วิภา", "กิตติ", "นภา", "อนันต์", "ปรียา", "ธีรพงษ์", "สุชาดา", "ณัฐวุฒิ", "กมลวรรณ",
               "พิชญ์", "ชลธิชา", "ธนกร", "วรัญญา", "ศรัณย์", "ภัทรพร", "อัครเดช", "กนกวรรณ", "ปัณณธร", "มณีรัตน์"]

last_names = ["ใจดี", "มีสุข", "วงศ์สว่าง", "เจริญรุ่ง", "แก้วมณี", "สุขสวัสดิ์", "พงษ์พาณิชย์", "รุ่งเรือง",
              "บุญมี", "สิริโชค", "วัฒนากุล", "เลิศรัตน์", "ทองแท้", "ปัญญายิ่ง", "รัตนไพศาล"]

tiers = ["Standard", "Silver", "Gold", "VIP"]

mock_books = [
    Book("B001", "ผ่าพิภพไททัน Vol.1", 90),
    Book("B002", "One Piece Vol.100", 95),
    Book("B003", "Jujutsu Kaisen Vol.1", 85),
    Book("B004", "คิดแบบสตีฟ จ็อบส์", 350),
    Book("B005", "Python Pro Guide", 450),
    Book("B006", "ดาบพิฆาตอสูร Vol.1", 85),
    Book("B007", "การตลาด 101", 250),
    Book("B008", "จิตวิทยาสายมืด", 290),
]

# ==========================================
# 3. สุ่มเลือกลูกค้า 1 คนจากลำดับ 1 - 300
# ==========================================
random_index = random.randint(1, 300)

name = f"{random.choice(first_names)} {random.choice(last_names)}"
c_id = f"C{random_index:03d}"
phone = f"08{random.randint(10000000, 99999999)}"
email = f"user{random_index}@example.com"
initial_points = random.randint(0, 500)
tier = random.choice(tiers)
reg_date = f"2025-{random.randint(1,12):02d}-{random.randint(1,28):02d}"

customer = LibraryOrder(name, c_id, phone, email, initial_points, tier, reg_date)

# สุ่มยืมหนังสือ 1 - 12 เล่ม
num_books = random.randint(1, 12)
rented_items = [random.choice(mock_books) for _ in range(num_books)]
days_rented = random.randint(1, 10)
days_late = 0 if random.random() > 0.25 else random.randint(1, 4)

# พิมพ์ใบเสร็จของคนที่ถูกสุ่มเพียงคนเดียว
print(f"🎯 ผลการสุ่มเลือกลูกค้าลำดับที่ {random_index} จาก 300 คน:\n")
receipt = Receipt(f"REC-2026-{random_index:03d}", customer, rented_items, days_rented, days_late)
receipt.print_receipt()

🎯 ผลการสุ่มเลือกลูกค้าลำดับที่ 243 จาก 300 คน:

                  ใบเสร็จรับเงิน / Receipt                  
เลขที่ใบเสร็จ: REC-2026-243         วันที่: 2026-08-26 15:39:19
ชื่อลูกค้า: อัครเดช ปัญญายิ่ง (ID: C243) | ระดับ: Standard
ระยะเวลายืม: 7 วัน | สถานะ: ✅ คืนตรงเวลา
------------------------------------------------------------
รายการหนังสือที่ยืม (3 เล่ม):
  [01] B005 - Python Pro Guide
  [02] B006 - ดาบพิฆาตอสูร Vol.1
  [03] B001 - ผ่าพิภพไททัน Vol.1
------------------------------------------------------------
ค่ายืมต่อเล่ม (7 วัน): 47.00 บาท
รวมค่ายืมหนังสือ: 141.00 บาท
------------------------------------------------------------
ยอดชำระสุทธิ (Grand Total): 141.00 บาท
✨ แต้มที่ได้รับ: +14 แต้ม | แต้มสะสมรวม: 315 แต้ม

